« model_19 — SABİT NOKTALAR ÜZERİNDE ÖĞRENİLEN İLİŞKİLER · HİKÂYE (TinyStories) · H1: MODEL DENEMELERİ »

**Soru: model_19'un yeni parçalarından hangisi hikâyede zemine göre ayırt edilebilir bir fark yapıyor?** Kullanıcı, 25 Eylül: *"aslında önce eğitim değil de diğer yapısal şeyleri denesek onlar daha kritik"*.
Tasarım `TASARIM_19.md`, karar kuralları koşudan önce `belge/onkayit/model_19.md` (HİKÂYE bölümü).

| ne | değer |
|---|---|
| veri | TinyStories v4 (`tam_n8000_v4`) · Drive önbelleğinden · her hikâye bir pencere, T 512 |
| model | D 1024 (sıra 512 + içerik 512) · 256 hareket, aktif 8, 4 katman · attention 4 × 64 · defter (bütün geçmiş) · rank 256 |
| eğitim | Adam, weight decay yok · batch 64 · LR 0,002 SABİT · 4.000 adım |
| ölçüm · kayıt | ölçüm ve ağırlık her 500 (2.000 + 2.000 pencere), tam yedek her 2.000 (decompose hikâye istemleriyle, döngü); bitişte tutulan pencerelerin TAMAMI |

| koşu | tohum | açık yeni parça |
|---|---|---|
| `PR_TS_ARCH_BASE_S0` · `_S1` · `_S2` | 0 · 1 · 2 | yok (zemin) |
| `PR_TS_ARCH_E` | 0 | embed (E) |
| `PR_TS_ARCH_RPC` | 0 | readout (R_PC) |
| `PR_TS_ARCH_DIST` | 0 | mesafe eğilimi |
| `PR_TS_ARCH_ATT2` | 0 | ikinci attention |
| `PR_TS_ARCH_RCC` | 0 | chain_sim (R_CC) |

**Karar:** kol tek tohum; |kol − zemin ortalaması| > zemin yayılımı ise **ADAY**. Aday için tohum 1 ve 2 onay ister, hüküm 3 tohumla. **Sağlama:** `BASE_S0`'ın 4.000 eğrisi TS_PV_V4'ün 0,4972'sinin yanına yazılır; 2 puandan fazla düşükse kollar başlamaz, kod incelenir.

**Sıra:** `0 HAZIRLIK` → `1a` (zemin, tohum 0: sağlama ve hız) → `4 EĞRİ` → `1b`–`1h` birer birer → `5 H1 KARARI` · `6 HİKÂYE YAZ` · `7 SONUÇ`
**Çekirdek düşerse:** `0 HAZIRLIK` → `2 SÜRDÜR`

**Dönen hücre YOK.** Koşu arka planda bir iplikte döner, her hücre hemen geri gelir (kural 8). **Koşular SIRAYLA** (torch.compile iş parçacığı güvenli değil); koşu hücresi başka canlı koşu varsa başlamaz. **Koşuları kullanıcı başlatır (kural 0).**

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "2 SURDUR".  Hikaye verisi ~4 GB RAM.
import os, re, sys, subprocess

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir ve SURDUR ikinci bir kopya baslatir.
if 'train_19' in sys.modules:
    _live = [r.name for r in sys.modules['train_19'].RUNS.values() if r.alive]
    assert not _live, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 3 NABIZ' % _live

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_19'
TS_DIR = '/content/drive/MyDrive/tinystories'
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sekerahmet/sekerai.git', REPO], check=True)
SRC = REPO + '/deneme2/model_19'
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
# Yerel commit GitHub'a gitmediyse burada durur, ESKI kodla kosmaz.
assert os.path.exists(SRC + '/data_stories_19.py'), 'depoda model_19 YOK -- yerelde git push gerekli'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_19', 'train_19', 'data_stories_19', 'decompose_19', 'diagnose_19', 'data_mat_19'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_19
import train_19 as train
import data_stories_19 as DS

# Kural 9: sozluk ve akis Drive onbelleginden; Colab veri URETMEZ.  v4: <nl>, kusur suzgeci, sozluk 8.000.
DATA, VOCAB_LIMIT = 'v4', 8000
_tag = 'tam_n%d_%s' % (VOCAB_LIMIT, DATA)
_cache = ['%s/onbellek/%s_%s.npy' % (TS_DIR, k, _tag) for k in ('sozluk', 'akis_train', 'akis_valid')]
assert all(os.path.exists(f) for f in _cache), 'onbellek %s Drive da YOK -- yerelde uretilir (kural 9)' % _tag
VOCAB, (_train_w, _train_m), (_heldout_w, _heldout_m) = DS.build(TS_DIR, T=512, limit=VOCAB_LIMIT, version=DATA)
TOKEN_ID = {a: i for i, a in enumerate(VOCAB)}
N, EOS = len(VOCAB), TOKEN_ID[DS.EOS_TOKEN]
TRAIN = (torch.from_numpy(_train_w), torch.from_numpy(_train_m))
HELDOUT = (torch.from_numpy(_heldout_w), torch.from_numpy(_heldout_m))
del _train_w, _train_m, _heldout_w, _heldout_m
# Saglik: her olcumde sabit sondada m.health; tam yedekte ve sonda sabit istemlerle dongu, metin ve decompose.
METRIC = DS.make_metric(TRAIN, HELDOUT, eos=EOS, device='cuda', limit=2000, vocab=VOCAB, prompts=DS.prompts(TS_DIR))

# Onkayit (belge/onkayit/model_19.md, HIKAYE): batch 64, 4.000 adim, sabit LR, olcum/agirlik 500, tam yedek 2.000.
COMMON = dict(batch=64, steps=4000, eval_every=500, save_every=2000, weights_every=500)
# Kullanici, 25 Eylul: "D 1024, 4×64".  rank 256 < D: iliskiler A B^T, E kelime basina U W^T.
STORY = dict(d_order=512, d_content=512, vectors=256, active=8, layers=4, attn_heads=4, attn_dim=64, t_max=512,
             rank=256)
BASE = dict(embed=False, readout=False, chain_sim=False, distance=False, attn_after=(0,))      # zemin
MODEL_BASE = dict(STORY, **BASE)

# IKI TUR DENEME AYRI IZLENIR.  Kullanici, 25 Eylul: "model içinde denediklerimiz ve eğitimde denediklerimiz ayrı
# şekilde takip etmeliyiz" ve "aslında önce eğitim değil de diğer yapısal şeyleri denesek onlar daha kritik".
# ONCE model denemeleri (H1, tarif sabit), egitim denemeleri (H2) sonra.  Ikisini birden degistiren kosu BASLAMAZ.
REF_RECIPE = dict(lr=0.002, warmup=0, decay_floor=0.1)    # model_18'in hikaye LR'si; model_19'da dogrulanmadi (kural 11)
TARIF = dict(REF_RECIPE)                                  # H1'in tarifi
TRAIN_TRIALS = {}                                         # H2: H1'den sonra kurulur
ARCH_TRIALS = {'PR_TS_ARCH_BASE_S%d' % s: (s, MODEL_BASE) for s in (0, 1, 2)}
for part, kw in (('E', dict(embed=True)), ('RPC', dict(readout=True)), ('DIST', dict(distance=True)),
                 ('ATT2', dict(attn_after=(0, 2))), ('RCC', dict(chain_sim=True))):
    ARCH_TRIALS['PR_TS_ARCH_' + part] = (0, dict(MODEL_BASE, **kw))

# ad -> (tur, egitim ayari, model ayari)
CONFIGS = {**{n: ('egitim', dict(tr, seed=0), MODEL_BASE) for n, tr in TRAIN_TRIALS.items()},
           **{n: ('model', dict(TARIF, seed=s), mk) for n, (s, mk) in ARCH_TRIALS.items()}}
# DAL: kaynak kosunun sogutma oncesi tam yedeginden, ayni ayarla, yeni planla (kaynak kayitlari yerinde kalir).
# Kullanici, 25 Eylul: "Bence 40.000 uzat tek epoch görelim".  H1 KARARI dallari okumaz.
# Kullanici, 25 Eylul: "Uzat ama önce Colab bağlantın var mı" -- 80K, 40K dalinin t32000'inden (sogutma oncesi).
BRANCHES = {'PR_TS_ARCH_BASE_S0_40K': ('PR_TS_ARCH_BASE_S0', 16000),      # ad -> (kaynak kosu, dal adimi)
            'PR_TS_ARCH_BASE_S0_80K': ('PR_TS_ARCH_BASE_S0_40K', 32000)}
for _b, (_src, _) in BRANCHES.items():                    # sirayla: bir dal baska bir dali kaynak alabilir
    CONFIGS[_b] = CONFIGS[_src]
EXTRA = dict(data='tinystories ' + _tag, fingerprint=DS.fingerprint(VOCAB, TRAIN[0][:1000].numpy()), T=512,
             code=CODE, onkayit='belge/onkayit/model_19.md')
TS_PV_V4_T4000 = 0.4972     # sağlama: model_18 TS_PV_V4'un 4.000 egrisi (baska kod, defter top-8) -- kiyas DEGIL
GPU_MARGIN = 1.2            # tahmin fazla: model_18'de tahmin 7,6 GB, nvidia-smi 5,2 GB (REL07, 24 Eylul)


def trial(name):
    '''-> (tur, referansa gore degisen).  Ikisini birden degistiren deneme DURUR.'''
    kind, tr, mk = CONFIGS[name]
    recipe = {k: tr.get(k, REF_RECIPE[k]) for k in REF_RECIPE}
    model = {k: v for k, v in mk.items() if MODEL_BASE.get(k) != v}
    if kind == 'egitim':
        assert not model, name + ': egitim denemesi modeli degistiremez -- ' + str(model)
        return kind, {k: v for k, v in recipe.items() if REF_RECIPE[k] != v}
    assert recipe == {k: TARIF.get(k, REF_RECIPE[k]) for k in REF_RECIPE}, name + ': model denemesi tarifi degistiremez'
    return kind, model


def memory_gb(d_order, d_content, vectors, active, layers, attn_heads, attn_dim, attn_after=(0,), **_):
    '''Ileri gecisin geri yayilim icin SAKLADIGI, TAHMIN (model_19'da olculmedi; T tam 512 sayilir, batch kirpilir):
    katman basina uzaklik tablosu, aktif hareketler ve nokta; N noktalik puan (x3) ve defterin karisimi (x4);
    defterin (T,T) benzerligi (x7, butun gecmis); icerik yarisinin parcalari (x6); attention basina q, k, v, cikti, (T,T).'''
    B, T, d = COMMON['batch'], TRAIN[0].shape[1], d_order + d_content
    attn = len(attn_after) * (4 * attn_heads * attn_dim + attn_heads * T)
    return B * T * (layers * (vectors + active * d + 3 * d) + 7 * N + 7 * T + 6 * d_content + attn) * 4 / 1e9


def launch(name, resume=None, steps=None, **plan):
    '''CONFIGS[name] ile arka planda baslatir (kural 8); GPU kapisi hucrenin kendisinde.  plan: uzatmada sogutma.'''
    kind, tr, mk = CONFIGS[name]
    kind, changed = trial(name)
    kw = dict(COMMON, **tr)
    kw.update(plan)                                       # plan tarifteki decay_floor'u ezebilir
    if steps is not None:
        kw['steps'] = steps
    return train.start(name, (TRAIN[0], TRAIN[1], TRAIN[1]), N, metric=METRIC, device='cuda', root=ROOT, vocab=VOCAB,
                       resume=resume, extra=dict(EXTRA, trial=kind, changed=changed), **kw, **mk)


def latest_model(name):
    '''Diskteki EN YENI agirlik (w ya da t) -- kosu surerken de, ayri bir kopya.'''
    d = ROOT + '/' + name
    f = max((int(re.findall('[0-9]+', x)[0]), x) for x in os.listdir(d) if re.match('[tw][0-9]+[.]pt$', x))[1]
    k = torch.load(d + '/' + f, weights_only=False, map_location='cuda')
    return model_19.PointRelation.from_package(k).cuda().eval(), k


n = len(TRAIN[0])
print('sozluk %s token   <eos> %d   egitim %s pencere   tutulan %s pencere   T=%d'
      % (f'{N:,}', EOS, f'{n:,}', f'{len(HELDOUT[0]):,}', TRAIN[0].shape[1]))
print('1 epok = %s adim   %s adim = %.2f epok' % (f"{n // COMMON['batch']:,}", f"{COMMON['steps']:,}",
                                                 COMMON['steps'] * COMMON['batch'] / n))
for kind_title, kind_key in (('MODEL DENEMELERI (H1, tarif = TARIF)', 'model'), ('EGITIM DENEMELERI (H2, model = zemin)', 'egitim')):
    names = [x for x in CONFIGS if CONFIGS[x][0] == kind_key]
    print('\n' + kind_title + ('' if names else '  -- henuz yok'))
    for name in names:
        _m = model_19.PointRelation(N, **CONFIGS[name][2])
        print('  %-22s tohum %d   degisen %-26s parametre %s   saklanan ~%.1f GB'
              % (name, CONFIGS[name][1]['seed'], trial(name)[1], f'{sum(p.numel() for p in _m.parameters()):,}',
                 memory_gb(**CONFIGS[name][2])))
del _m
print('\nORNEK PENCERE\n' + DS.decode(TRAIN[0][0][TRAIN[1][0]].numpy(), VOCAB)[:600])

In [ ]:
# 1a KOSU -- H1 zemin, tohum 0 (ILK: saglama ve hiz)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_BASE_S0'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1b KOSU -- H1 kol E (embed)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_E'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1c KOSU -- H1 kol R_PC (readout)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_RPC'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1d KOSU -- H1 kol mesafe egilimi (distance)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_DIST'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1e KOSU -- H1 kol ikinci attention (attn_after 0, 2)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_ATT2'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1f KOSU -- H1 kol R_CC (chain_sim)  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_RCC'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1g KOSU -- H1 zemin, tohum 1  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_BASE_S1'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1h KOSU -- H1 zemin, tohum 2  |  GPU  |  tekrar: degil -- ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
NAME = 'PR_TS_ARCH_BASE_S2'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()           # biten kosunun onbellegi bos sayilsin
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(launch(NAME))

In [ ]:
# 1i DAL -- H1 zemin tohum 0, t16000'dan 40.000'e (sogutma 32.000 -> 40.000)  |  GPU  |  tekrar: degil -- ayni adla
#     ikinci kez calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# Kullanici, 25 Eylul: "Bence 40.000 uzat tek epoch görelim".  t16000'da sogutma baslamamisti (LR sabit): 16.000'e
# kadar yorunge BASE_S0 ile birebir.  1 epok = 41.602 adim; ornekleme yerine koymali (randint).
NAME = 'PR_TS_ARCH_BASE_S0_40K'
STEPS = 40000
PLAN = dict(decay_start=32000, decay_floor=0.1)     # son %20, lr -> lr/10 (BASE_S0'daki oran)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc, os
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

_src, _at = BRANCHES[NAME]
RESUME_FROM = '%s/%s/t%d.pt' % (ROOT, _src, _at)
assert os.path.exists(RESUME_FROM), 'dal paketi YOK -- ' + RESUME_FROM
print('dal  %s   adim %s -> %s   plan %s' % (RESUME_FROM, f'{_at:,}', f'{STEPS:,}', PLAN))
print(launch(NAME, resume=RESUME_FROM, steps=STEPS, **PLAN))

In [ ]:
# 1j DAL -- H1 zemin tohum 0, 40K dalinin t32000'inden 80.000'e (sogutma 64.000 -> 80.000)  |  GPU  |  tekrar: degil -- ayni adla
#     ikinci kez calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# Kullanici, 25 Eylul: "Uzat ama önce Colab bağlantın var mı gitmiş olabilir".  t32000'de sogutma baslamamisti
# (LR sabit): 32.000'e kadar yorunge 40K daliyla birebir.  80.000 adim ~1,9 epoklik ornek.
NAME = 'PR_TS_ARCH_BASE_S0_80K'
STEPS = 80000
PLAN = dict(decay_start=64000, decay_floor=0.1)     # son %20, lr -> lr/10 (ayni oran)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import gc, os
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

_src, _at = BRANCHES[NAME]
RESUME_FROM = '%s/%s/t%d.pt' % (ROOT, _src, _at)
assert os.path.exists(RESUME_FROM), 'dal paketi YOK -- ' + RESUME_FROM
print('dal  %s   adim %s -> %s   plan %s' % (RESUME_FROM, f'{_at:,}', f'{STEPS:,}', PLAN))
print(launch(NAME, resume=RESUME_FROM, steps=STEPS, **PLAN))

In [ ]:
# 2 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.  NAME'i sec.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin STEPS'i buyut, bu hucreyi calistir.
import gc, os, re, torch
NAME = 'PR_TS_ARCH_BASE_S0'     # CONFIGS'teki adlardan biri
STEPS = 20000                   # hedef; uzatmada buyut (kural 1)
# Kullanici, 25 Eylul: "bence uzatmayı 20.000 yap ordan bakalım 4.000 çok erken karar için".  Sogutma son %20.
PLAN = dict(decay_start=16000, decay_floor=0.1)     # kosunun LR plani; train_19._check_resume denetler

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_live = [r.name for r in train.RUNS.values() if r.alive]
assert not _live, 'kosu suruyor: %s -- kosular SIRAYLA (torch.compile)' % _live
gc.collect()
torch.cuda.empty_cache()
_free = torch.cuda.mem_get_info()[0] / 1e9
_need = max(2.0, GPU_MARGIN * memory_gb(**CONFIGS[NAME][2]))
assert _free > _need, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_free, _need)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB' % (torch.cuda.get_device_name(0), _free, _need))

_d = ROOT + '/' + NAME
_n = sorted((int(re.findall('[0-9]+', f)[0]), f) for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
RESUME_FROM = _d + '/' + _n[-1][1]
print('son nokta  %s   adim %s   hedef %s' % (RESUME_FROM, f'{_n[-1][0]:,}', f'{STEPS:,}'))
assert _n[-1][0] < STEPS, 'zaten hedefe varmis -- uzatmak icin STEPS buyut'
print(launch(NAME, resume=RESUME_FROM, steps=STEPS, **PLAN))

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu ve GPU bellek tepesini basar.
# Bellekteki BUTUN kosu nesneleri (HAZIRLIK modulleri yeniden yuklese de eski kosu gorunur): canlilarin son 30
# satiri, bitenlerin son satiri.
import gc
import torch
_runs = [o for o in gc.get_objects() if type(o).__name__ == 'Run']
for _r in sorted(_runs, key=lambda r: not r.alive):
    print('%s: %s   gunluk %d satir%s' % (_r.name, 'CANLI' if _r.alive else 'bitti', len(_r.log),
                                          '   DURDUR istendi' if _r.stop_requested else ''))
    for _s in (_r.log[-30:] if _r.alive else _r.log[-1:]):
        print(_s)
if torch.cuda.is_available():
    print('\nGPU bellek tepesi (bu cekirdekte) %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))

In [ ]:
# 4 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Ust: heldout accuracy (her 500 adim, 2.000 pencere).  Alt: HER ADIMIN kaybi.  Saglama: BASE_S0'in 4.000'i.
import os
import torch
import matplotlib.pyplot as plt


def log_points(name):
    '''gunluk.txt -> {adim: (kayip, train, heldout)}.'''
    r, path = {}, ROOT + '/' + name + '/gunluk.txt'
    if os.path.exists(path):
        for s in open(path, encoding='utf-8'):
            p = s.split()
            if len(p) >= 6 and p[1].isdigit():
                try:
                    r[int(p[1])] = tuple(float(x) for x in p[2:5])
                except ValueError:
                    pass
    return r


def step_losses(name):
    '''Canli kosudan (train.RUNS) ya da diskteki son paketten.'''
    k = train.RUNS[name].result.get('step_losses') if name in train.RUNS else None
    path = ROOT + '/model_' + name + '.pt'
    if k is None and os.path.exists(path):
        try:
            k = torch.load(path, weights_only=False, map_location='cpu').get('step_losses')
        except Exception as h:          # yazilirken okunduysa
            print('  %s paketi okunamadi (%s) -- tekrar dene' % (name, h))
    return None if k is None else k[~k.isnan()]


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for name in CONFIGS:
    r, k = log_points(name), step_losses(name)
    if r:
        x = sorted(r)
        a1.plot(x, [r[i][2] for i in x], label=name)
        print('%-22s %d nokta   son adim %s   heldout %.4f' % (name, len(x), f'{x[-1]:,}', r[x[-1]][2]))
    if k is not None and len(k) > 200:
        a2.plot(k.numpy(), lw=0.4, label=name)
a1.axhline(TS_PV_V4_T4000, ls=':', c='gray', label='TS_PV_V4 t4000 (model_18, saglama)')
r = log_points('PR_TS_ARCH_BASE_S0')
if 4000 in r:
    d = 100 * (r[4000][2] - TS_PV_V4_T4000)
    print('\nSAGLAMA  BASE_S0 t4000 egri %.4f   TS_PV_V4 %.4f   fark %+.2f puan   %s'
          % (r[4000][2], TS_PV_V4_T4000, d, 'KOLLAR BASLAMAZ, kod incelenir (onkayit)' if d < -2 else 'gecti'))
a1.set_ylabel('heldout accuracy')
a1.legend(fontsize=7, ncol=2)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 H1 KARARI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar; kural belge/onkayit/model_19.md (HIKAYE, H1), aynen
# Zemin: 3 tohumun bitisteki tam olcumu -> ortalama m, yayilim y.  Kol (tek tohum): |kol - m| > y ise ADAY;
# aday icin tohum 1 ve 2 ONAY ister, hukum 3 tohumla.  Iraksayan (NaN ya da son 1.000 adim ilk 500'den kotu) ARIZA.
import math, os, re, torch


def final(name):
    '''Ilk BITTI satiri (4.000'deki tam olcum) -> (heldout, ppl); bitmediyse None.'''
    path = ROOT + '/' + name + '/gunluk.txt'
    for s in (open(path, encoding='utf-8') if os.path.exists(path) else ()):
        if ' BITTI ' in s:
            return float(re.search(r'heldout ([0-9.]+)', s).group(1)), float(re.search(r'ppl ([0-9.]+)', s).group(1))
    return None


def diverged(name):
    k = torch.load(ROOT + '/model_' + name + '.pt', weights_only=False, map_location='cpu').get('step_losses')
    k = k[~k.isnan()]
    return (not math.isfinite(float(k[-1]))) or float(k[-1000:].mean()) > float(k[:500].mean())


base = {n: final(n) for n in ARCH_TRIALS if n.startswith('PR_TS_ARCH_BASE_')}
for n, r in base.items():
    print('%-22s %s' % (n, 'heldout %.4f   ppl %.2f%s' % (r + ('   IRAKSADI' if diverged(n) else '',)) if r else 'BITMEDI'))
done = [r[0] for r in base.values() if r]
m = y = None
if len(done) == 3:
    m, y = sum(done) / 3, max(done) - min(done)
    print('\nzemin ortalamasi m %.4f   yayilim y %.4f (%.2f puan)\n' % (m, y, 100 * y))
else:
    print('\nzemin %d / 3 tohum -- hukum icin uc tohum gerekir\n' % len(done))
for n in ARCH_TRIALS:
    if n.startswith('PR_TS_ARCH_BASE_'):
        continue
    r = final(n)
    if not r:
        print('%-22s BITMEDI' % n)
        continue
    line = '%-22s heldout %.4f   ppl %.2f' % (n, r[0], r[1])
    if diverged(n):
        line += '   IRAKSADI -- ARIZA: tohum eklenmez, decompose ile incelenir'
    elif m is not None:
        d = r[0] - m
        line += '   fark %+.2f puan   %s' % (100 * d, 'ADAY (%s) -- tohum 1 ve 2 ONAY ister' % '+-'[d < 0]
                                           if abs(d) > y else 'ayirt edilemedi (tek tohum)')
    print(line)
print('\nR_CC (Oe5): heldout artarken uzak kalip artiyorsa TAKAS -- gunlukteki decompose satirlari ve metin gozle.')

In [ ]:
# 6 HIKAYE YAZ  |  GPU  |  ARKA PLAN: ilk calistirma baslatir, sonrakiler okur  |  tekrar: GUVENLI
# SAYI degil METIN (kural 12): model istemin devamini yazar, <eos> uretince durur.  Diskteki EN YENI agirlikla.
NAME = 'PR_TS_ARCH_BASE_S0'
N_PROMPTS, TEMPERATURES = 4, (0.0, 0.8)
REDO = False                    # True: bitmis isi diskteki yeni agirlikla bastan yazdir

import random, textwrap, threading
JOBS = globals().setdefault('JOBS', {})      # arka plan isleri (train.Run: NABIZ da gorur)
_run_name = 'HIKAYE ' + NAME
if _run_name not in JOBS or (REDO and not JOBS[_run_name].alive):
    # --- GPU KAPISI (CLAUDE.md kural 2)
    assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'

    def _story_job(r, name=NAME):
        try:
            m, k = latest_model(name)
            r.note('%s   adim %s   heldout accuracy %.4f' % (name, f"{k['step']:,}", k['heldout_acc']))
            for prompt in random.Random(0).sample(DS.prompts(TS_DIR), N_PROMPTS):
                for s in TEMPERATURES:
                    prompt_text, generated = DS.generate(m, prompt, VOCAB, TOKEN_ID, steps=150, device='cuda',
                                                         temperature=s, seed=0)
                    r.note('-' * 72)
                    r.note('ISTEM   ' + '\n        '.join(textwrap.wrap(prompt_text, 64)))
                    r.note('MODEL (sicaklik %.1f)\n        ' % s + '\n        '.join(textwrap.wrap(generated, 64)))
        except Exception as h:
            r.note('HATA  %r' % h)

    JOBS[_run_name] = train.Run(_run_name)
    JOBS[_run_name].thread = threading.Thread(target=_story_job, args=(JOBS[_run_name],), daemon=True)
    JOBS[_run_name].thread.start()
_r = JOBS[_run_name]
print('%s: %s' % (_run_name, 'YAZIYOR -- hucreyi tekrar calistir' if _r.alive else 'bitti'))
for _s in _r.log:
    print(_s.split('] ', 1)[1])

In [ ]:
# 7 SONUC  |  GPU  |  ARKA PLAN: ilk calistirma baslatir, sonrakiler okur  |  tekrar: GUVENLI
# Kosu BITTIKTEN sonra: tutulan pencerelerin TAMAMINDA konuma gore accuracy.
import math, threading
NAME = 'PR_TS_ARCH_BASE_S0'
REDO = False                    # True: bitmis olcumu diskteki yeni agirlikla bastan yap

JOBS = globals().setdefault('JOBS', {})      # arka plan isleri (train.Run: NABIZ da gorur)
_run_name = 'SONUC ' + NAME
if _run_name not in JOBS or (REDO and not JOBS[_run_name].alive):
    # --- GPU KAPISI (CLAUDE.md kural 2)
    assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
    assert not (NAME in train.RUNS and train.RUNS[NAME].alive), NAME + ' HALA KOSUYOR'

    def _result_job(r, name=NAME):
        try:
            m, k = latest_model(name)
            r.note('%s   adim %s   (TAM) train %.4f   heldout %.4f   ppl %.2f'
                   % (name, f"{k['step']:,}", k['train_acc'], k['heldout_acc'], math.exp(k['heldout_ce']))
                   if k.get('done') else '%s   adim %s  -- BITMEMIS' % (name, f"{k['step']:,}"))
            r.note('KONUMA GORE accuracy (tutulan): C hikayenin neresinde doyuyor')
            DS.position_table(m, HELDOUT[0], HELDOUT[1], device='cuda', log=r.note)
        except Exception as h:
            r.note('HATA  %r' % h)

    JOBS[_run_name] = train.Run(_run_name)
    JOBS[_run_name].thread = threading.Thread(target=_result_job, args=(JOBS[_run_name],), daemon=True)
    JOBS[_run_name].thread.start()
_r = JOBS[_run_name]
print('%s: %s' % (_run_name, 'OLCUYOR -- hucreyi tekrar calistir' if _r.alive else 'bitti'))
for _s in _r.log:
    print(_s.split('] ', 1)[1])

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
train.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for name in CONFIGS:
    _d = ROOT + '/' + name
    if not os.path.isdir(_d):
        print('%-22s henuz kayit yok' % name)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f if os.path.isfile(_d + '/' + f))
    print('%-22s %3d yedek   %.3f GB   %s' % (name, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi